# Abstract Classes in Python

# 1. Why do we need abstract classes?

In the previous seminar, the class `Animal` had the method `speak`, which only raised `NotImplementedError`. This is the "old" way to describe a common interface for child classes. Let us look at its problems with a new example.

A program collects data from different sensors. Every sensor must have the method `read()`, which returns the current value. The parent class `Sensor` also has the method `report()`, which uses `read()`:

In [ ]:
class Sensor:
    def __init__(self, name):
        self.name = name

    def read(self):
        raise NotImplementedError('a child class must implement read()')

    def report(self):
        return f'{self.name}: {self.read()}'


class Thermometer(Sensor):
    def read(self):
        return 21.5


class Barometer(Sensor):
    def reed(self):          # a typo in the name of the method!
        return 101.3

In [ ]:
sensors = [Thermometer('t1'), Barometer('b1'), Sensor('unknown')]
print('All the sensors are created')

for sensor in sensors:
    print(sensor.report())

All the sensors are created
t1: 21.5


NotImplementedError: a child class must implement read()

This code has three problems:
- Python allows us to create an object of the class `Sensor`, although a "sensor in general" cannot measure anything.
- The typo `reed` gives no error when we create the class `Barometer` or its object.
- We see the error only when we **call** the method. In a real program, this can happen after an hour of work, or only on the computer of a user.

We want Python to find such errors **as early as possible**, when we create an object. This is the job of abstract classes.

**An abstract class** is a class that describes a common interface for its child classes, but we cannot create instances of it. It contains **abstract methods**: methods that have a name, but every child class must give its own implementation. A child class that implements all the abstract methods is called a **concrete class**.

We can think of an abstract class as a **contract**: "if you inherit from me, you must have these methods".

In Python, abstract classes are in the standard module [`abc`](https://docs.python.org/3/library/abc.html) (**A**bstract **B**ase **C**lasses).

# 2. `ABC` and `@abstractmethod`

To create an abstract class, we need two steps:
1. the class inherits from the class `ABC`;
2. the abstract methods are marked with the decorator `@abstractmethod`.

In [ ]:
from abc import ABC, abstractmethod


class Sensor(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def read(self):
        """Return the current value of the sensor."""

    def report(self):
        return f'{self.name}: {self.read()}'

The body of an abstract method is often only a **docstring** that explains what the method must do. We can also write `pass`, `...` or `raise NotImplementedError` (as in the previous seminar). All these variants work, but a docstring is the most useful one: it is the documentation of the contract.

Now let us try to create an object of the abstract class:

In [ ]:
try:
    Sensor('unknown')
except TypeError as error:
    print(error)

Can't instantiate abstract class Sensor without an implementation for abstract method 'read'


The word *instantiate* means "create an instance". Now let us repeat our example with the typo:

In [ ]:
class Thermometer(Sensor):
    def read(self):
        return 21.5


class Barometer(Sensor):
    def reed(self):          # the same typo
        return 101.3


t = Thermometer('t1')
print(t.report())

b = Barometer('b1')

t1: 21.5


TypeError: Can't instantiate abstract class Barometer without an implementation for abstract method 'read'

Now the error appears **at once**, when we try to create the object, and the message names the missing method. The class `Barometer` has not implemented `read`, so it is still an abstract class.

Note that Python checks the class when we create an **object**, not when we create the **class**. The definition of the class `Barometer` gives no error.

### `__abstractmethods__` and `inspect.isabstract`

Python stores the names of the abstract methods that are not implemented yet in the attribute `__abstractmethods__`. For a concrete class, this set is empty. The function `inspect.isabstract` checks whether a class is abstract:

In [ ]:
import inspect

for cls in [Sensor, Barometer, Thermometer]:
    print(f'{cls.__name__:>11}: {cls.__abstractmethods__}, abstract: {inspect.isabstract(cls)}')

     Sensor: frozenset({'read'}), abstract: True
  Barometer: frozenset({'read'}), abstract: True
Thermometer: frozenset(), abstract: False


### An abstract class is still a normal parent class

We cannot create instances of an abstract class, but it works as a parent class in the usual way: `isinstance` and `issubclass` take it into account. Therefore, we can write general code for **all** sensors, including the sensors that we will add in the future:

In [ ]:
class Hygrometer(Sensor):
    def read(self):
        return 45


sensors = [Thermometer('t1'), Hygrometer('h1')]
for sensor in sensors:
    print(sensor.report(), '|', isinstance(sensor, Sensor))

t1: 21.5 | True
h1: 45 | True


This is subtype polymorphism from the previous seminar. The difference is that now Python **guarantees** that every object in the list has the method `read()`.

### A hierarchy of abstract classes

A child class can implement only some of the abstract methods. Then it is also abstract. This is useful when we build a hierarchy with several levels: the middle classes add common code, and only the classes at the bottom are concrete.

In [ ]:
class Animal(ABC):
    @abstractmethod
    def speak(self):
        """Return the sound of the animal."""

    @abstractmethod
    def move(self):
        """Return how the animal moves."""


class Bird(Animal):              # implements only move(), so it is still abstract
    def move(self):
        return 'flies'


class Parrot(Bird):              # implements the last abstract method
    def speak(self):
        return 'Hello!'


for cls in [Animal, Bird, Parrot]:
    print(f'{cls.__name__:>6}: abstract = {inspect.isabstract(cls)}, missing = {sorted(cls.__abstractmethods__)}')

kesha = Parrot()
print(kesha.speak(), kesha.move())

Animal: abstract = True, missing = ['move', 'speak']
  Bird: abstract = True, missing = ['speak']
Parrot: abstract = False, missing = []
Hello! flies


**Summary:**
- An abstract class inherits from `ABC`; its abstract methods are marked with `@abstractmethod`.
- We cannot create an instance of a class if at least one of its abstract methods is not implemented. Python raises `TypeError` and names the missing methods.
- The check happens when we create an object, not when we call a method.
- `Class.__abstractmethods__` contains the names of the methods that are not implemented yet; `inspect.isabstract(Class)` returns `True` for an abstract class.
- A child class that does not implement all the abstract methods is also abstract.

# 3. What can an abstract class contain?

An abstract class is not only a list of abstract methods. It can contain everything that a normal class contains:
- an initializer `__init__` and attributes (see the class `Sensor`: we cannot create a "sensor in general", but its `__init__` works for all the child classes);
- normal (concrete) methods; they can call the abstract methods (see `Sensor.report()`);
- abstract methods **with a body**.

### An abstract method with a body

The decorator `@abstractmethod` means "every child class must override this method". It does **not** mean "this method has no code". An abstract method can contain the common part of the work, and the child classes call it with `super()`:

In [ ]:
class Document(ABC):
    def __init__(self, title):
        self.title = title

    @abstractmethod
    def render(self):
        # the common part for all documents
        return f'=== {self.title} ==='


class Letter(Document):
    def __init__(self, title, recipient, text):
        super().__init__(title)
        self.recipient = recipient
        self.text = text

    def render(self):
        header = super().render()          # use the code of the abstract method
        return f'{header}\nDear {self.recipient},\n{self.text}'


class Invoice(Document):
    def __init__(self, title, amount):
        super().__init__(title)
        self.amount = amount

    def render(self):
        return f'{super().render()}\nTo pay: {self.amount} RUB'


print(Letter('Invitation', 'Anna', 'Welcome to the seminar!').render())
print()
print(Invoice('Invoice No. 15', 1200).render())

=== Invitation ===
Dear Anna,
Welcome to the seminar!

=== Invoice No. 15 ===
To pay: 1200 RUB


Every child class **must** write its own method `render`, but it does not need to repeat the common code.

### The template method

Very often, a concrete method of an abstract class describes the **steps of an algorithm**, and the abstract methods are the steps that are different in each child class. This idea is called the **template method** pattern: the parent class gives a template, and the child classes fill in the gaps.

Let us take an example from numerical methods: numerical integration. A composite quadrature rule splits the interval $[a, b]$ into $n$ equal segments of length $h = (b - a)/n$ and adds the results for all segments. The rules differ only in the formula for one segment $[x_i, x_{i+1}]$:

| Rule | The integral over one segment |
|---|---|
| midpoint rule | $h\, f\!\left(\frac{x_i + x_{i+1}}{2}\right)$ |
| trapezoid rule | $h\, \frac{f(x_i) + f(x_{i+1})}{2}$ |
| Simpson's rule | $\frac{h}{6}\left(f(x_i) + 4 f\!\left(\frac{x_i + x_{i+1}}{2}\right) + f(x_{i+1})\right)$ |

So the loop over the segments is the template, and the formula for one segment is an abstract method:

In [ ]:
import math


class QuadratureRule(ABC):
    def integrate(self, f, a, b, n=10):
        """The template method: the same algorithm for all the rules."""
        if n <= 0:
            raise ValueError('n must be positive')
        h = (b - a) / n
        total = 0
        for i in range(n):
            left = a + i * h
            total += self.segment(f, left, left + h)
        return total

    @abstractmethod
    def segment(self, f, left, right):
        """Return the integral of f over one segment [left, right]."""


class MidpointRule(QuadratureRule):
    def segment(self, f, left, right):
        return (right - left) * f((left + right) / 2)


class TrapezoidRule(QuadratureRule):
    def segment(self, f, left, right):
        return (right - left) * (f(left) + f(right)) / 2


class SimpsonRule(QuadratureRule):
    def segment(self, f, left, right):
        middle = (left + right) / 2
        return (right - left) * (f(left) + 4 * f(middle) + f(right)) / 6

In [ ]:
exact = 2.0   # the integral of sin(x) from 0 to pi

for rule in [MidpointRule(), TrapezoidRule(), SimpsonRule()]:
    result = rule.integrate(math.sin, 0, math.pi, n=10)
    print(f'{type(rule).__name__:>13}: {result:.8f}, error = {abs(result - exact):.1e}')

 MidpointRule: 2.00824841, error = 8.2e-03
TrapezoidRule: 1.98352354, error = 1.6e-02
  SimpsonRule: 2.00000678, error = 6.8e-06


The loop over the segments is written **only once**, in the abstract class. To add a new rule, we write one small method. The template method also makes all the rules work in the same way: for example, all of them check the value of `n`.

**Summary:**
- An abstract class can have an initializer, attributes and normal methods.
- An abstract method can have a body. The child class must override it, but it can call the parent version with `super()`.
- **Template method:** a normal method of the abstract class describes the algorithm, and the abstract methods are its steps.

**Check yourself.** What will the following code print? Think first, then run the cell.

In [ ]:
class Base(ABC):
    @abstractmethod
    def hello(self):
        return 'Hello from Base'


class Child(Base):
    def hello(self):
        return super().hello() + ' and from Child'


class Lazy(Base):
    pass


print(Child().hello())
print(Lazy.__abstractmethods__)
print(Lazy())

Hello from Base and from Child
frozenset({'hello'})


TypeError: Can't instantiate abstract class Lazy without an implementation for abstract method 'hello'

# 4. Abstract properties, class methods and static methods

We can combine `@abstractmethod` with the decorators that we already know: `@property`, `@classmethod` and `@staticmethod`.

**The rule:** `@abstractmethod` must be the **innermost** decorator, that is, the decorator that is closest to `def`. Decorators are applied from the bottom to the top, so `@abstractmethod` marks the function first, and then the other decorator wraps it.

### Abstract properties

An abstract property means: "every child class must give this **value**". For example, every vehicle has some number of wheels:

In [ ]:
class Vehicle(ABC):
    @property
    @abstractmethod
    def wheels(self):
        """The number of wheels."""

    def describe(self):
        return f'{type(self).__name__}: {self.wheels} wheels'


class Bicycle(Vehicle):
    @property
    def wheels(self):
        return 2


class Car(Vehicle):
    wheels = 4          # a simple class attribute also implements the abstract property


for vehicle in [Bicycle(), Car()]:
    print(vehicle.describe())

Vehicle()

Bicycle: 2 wheels
Car: 4 wheels


TypeError: Can't instantiate abstract class Vehicle without an implementation for abstract method 'wheels'

Note the class `Car`. When Python checks a child class, it only checks that the name `wheels` now refers to something that is **not abstract**. A class attribute is enough. This is a simple way to implement an abstract property if the value is a constant.

### Abstract class methods

In the previous seminar, we used class methods as **alternative constructors**, for example `Phone.toy_phone('Red')`. An abstract class method is a good way to say: "every child class must know how to create its objects from some data". For example, from a line of a text file:

In [ ]:
class Record(ABC):
    @classmethod
    @abstractmethod
    def from_line(cls, line):
        """Create an object from one line of a text file."""

    @classmethod
    def from_lines(cls, lines):
        # a normal class method uses the abstract one
        return [cls.from_line(line) for line in lines]


class Point(Record):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    @classmethod
    def from_line(cls, line):
        x, y = line.split(',')
        return cls(float(x), float(y))

    def __repr__(self):
        return f'Point({self.x}, {self.y})'


points = Point.from_lines(['1,2', '3.5,-1', '0,0'])
print(points)

[Point(1.0, 2.0), Point(3.5, -1.0), Point(0.0, 0.0)]


The method `from_lines` is written once, in the abstract class. Inside it, `cls` is the real class (`Point`), so `cls.from_line` calls the implementation of the child class.

### Abstract static methods

An abstract static method is less common. It says that every child class must have a helper function with this name:

In [ ]:
class Validator(ABC):
    @staticmethod
    @abstractmethod
    def is_valid(value):
        """Return True if the value is correct."""


class PositiveNumber(Validator):
    @staticmethod
    def is_valid(value):
        return isinstance(value, (int, float)) and value > 0


print(PositiveNumber.is_valid(5), PositiveNumber.is_valid(-1), PositiveNumber.is_valid('5'))

True False False


### The wrong order of decorators

If we put `@abstractmethod` on top, Python raises an error already when it creates the class:

In [ ]:
try:
    class Broken(ABC):
        @abstractmethod          # wrong: abstractmethod must be the closest to def
        @property
        def value(self):
            """Some value."""
except AttributeError as error:
    print('AttributeError:', error)

AttributeError: attribute '__isabstractmethod__' of 'property' objects is not writable


**Summary: the order of decorators**

| What we need | Decorators (from top to bottom) |
|---|---|
| an abstract method | `@abstractmethod` |
| an abstract property | `@property`, then `@abstractmethod` |
| an abstract class method | `@classmethod`, then `@abstractmethod` |
| an abstract static method | `@staticmethod`, then `@abstractmethod` |



# 5. What Python checks and what it does not check

An abstract class checks **only one thing**: when we create an object, every abstract name must be replaced with something that is not abstract. Python does **not** check:
- the parameters of the method;
- the type of the return value;
- that the new "method" is a method at all (remember `Car.wheels = 4`).

In [ ]:
class Converter(ABC):
    @abstractmethod
    def convert(self, value):
        """Convert one value and return the result."""


class Doubler(Converter):
    def convert(self):          # a wrong list of parameters
        return 'nothing'


d = Doubler()                   # no error here!
print('The object is created')
d.convert(5)                    # the error appears only here

The object is created


TypeError: Doubler.convert() takes 1 positional argument but 2 were given

So an abstract class is a contract for the **names** of methods, not for their full signatures. Code editors and type checkers can find such mistakes with the help of type hints, but Python itself does not.

### Typical mistakes

**1. `@abstractmethod` without `ABC`.** The decorator only puts a mark on the function. The check works only in classes that inherit from `ABC` (more exactly, in classes with the metaclass `ABCMeta`, see the next part). Without `ABC`, Python ignores the mark:

In [ ]:
class NotReallyAbstract:          # we forgot to inherit from ABC
    @abstractmethod
    def run(self):
        """Run the task."""


obj = NotReallyAbstract()         # no error
print(obj.run())                  # the "abstract" method runs and returns None

None


**2. `ABC` without abstract methods.** A class that inherits from `ABC` but has no abstract methods is not abstract: we can create its instances. `ABC` alone does not forbid anything.

In [ ]:
class Config(ABC):
    pass


config = Config()                 # no error
print(isinstance(config, Config), inspect.isabstract(Config))

True False


**3. Calling the "empty" abstract method with `super()`.** If the body of the abstract method is only a docstring, `super().method()` returns `None`. Call the parent version only if it really contains code.

**Summary:**
- Python checks only the **names** of the abstract methods, and only when we create an object.
- Python does not check the parameters, the return values or the types of the implementations.
- `@abstractmethod` works only in a class that inherits from `ABC`.
- A class that inherits from `ABC` without abstract methods is a normal class.

# 6.



### The abstract classes of `collections.abc`

The module [`collections.abc`](https://docs.python.org/3/library/collections.abc.html) contains abstract classes for containers. Here are some of them:

| Abstract class | Abstract methods | Methods that we get for free | Examples |
|---|---|---|---|
| `Sized` | `__len__` | — | all containers |
| `Iterable` | `__iter__` | — | `list`, `str`, `range`, generators |
| `Container` | `__contains__` | — | `list`, `set`, `dict` |
| `Collection` | `__len__`, `__iter__`, `__contains__` | — | `list`, `set`, `dict` |
| `Sequence` | `__getitem__`, `__len__` | `__contains__`, `__iter__`, `__reversed__`, `index`, `count` | `list`, `tuple`, `str`, `range` |
| `Mapping` | `__getitem__`, `__iter__`, `__len__` | `__contains__`, `keys`, `items`, `values`, `get`, `__eq__`, `__ne__` | `dict` |

So these classes have two uses:
1. **for checks**: `isinstance(x, Iterable)` is true for every iterable object, not only for lists;
2. **as parent classes**: we implement a few abstract methods and get the other methods for free (they are called **mixin methods**).

Let us create a sequence of squares $0, 1, 4, 9, \dots$ that does not store the numbers. We implement only two methods:

In [ ]:
from collections.abc import Sequence


class Squares(Sequence):
    """The sequence 0, 1, 4, 9, ... of n squares. It does not store the numbers."""

    def __init__(self, n):
        self.n = n

    def __len__(self):
        return self.n

    def __getitem__(self, index):
        # we do not support slices here, to keep the example short
        if index < 0:
            index += self.n
        if not 0 <= index < self.n:
            raise IndexError('index out of range')
        return index ** 2


sq = Squares(6)
print(len(sq), sq[2], sq[-1])
print(list(sq))                    # __iter__ from Sequence
print(16 in sq, 15 in sq)          # __contains__ from Sequence
print(list(reversed(sq)))          # __reversed__ from Sequence
print(sq.index(9), sq.count(4))    # index and count from Sequence

6 4 25
[0, 1, 4, 9, 16, 25]
True False
[25, 16, 9, 4, 1, 0]
3 1


We wrote two methods and got seven. If we forget one of the abstract methods, Python tells us at once:

In [ ]:
class BrokenSquares(Sequence):
    def __getitem__(self, index):
        return index ** 2


BrokenSquares()

TypeError: Can't instantiate abstract class BrokenSquares without an implementation for abstract method '__len__'

### The numeric tower: the module `numbers`

The module [`numbers`](https://docs.python.org/3/library/numbers.html) contains a hierarchy of abstract classes for numbers. It is called the **numeric tower**:

$$\texttt{Number} \supset \texttt{Complex} \supset \texttt{Real} \supset \texttt{Rational} \supset \texttt{Integral}$$

In the previous seminar, the function `double_it` used `Number` to accept all numbers (see also the additional reading). With the tower, we can choose a more exact level. Let us see where the built-in types are:

In [ ]:
import numbers
from fractions import Fraction
from decimal import Decimal

tower = [numbers.Integral, numbers.Rational, numbers.Real, numbers.Complex, numbers.Number]
values = [True, 7, Fraction(1, 3), 2.5, Decimal('1.1'), 1 + 2j]

for value in values:
    levels = [cls.__name__ for cls in tower if isinstance(value, cls)]
    print(f'{type(value).__name__:>8}: {levels}')

    bool: ['Integral', 'Rational', 'Real', 'Complex', 'Number']
     int: ['Integral', 'Rational', 'Real', 'Complex', 'Number']
Fraction: ['Rational', 'Real', 'Complex', 'Number']
   float: ['Real', 'Complex', 'Number']
 Decimal: ['Number']
 complex: ['Complex', 'Number']


Note that `Decimal` is only a `Number`, not a `Real`. The reason is that `Decimal` does not work together with `float`: `Decimal('1.1') + 2.5` raises `TypeError`.

A function that needs a real number can check `numbers.Real`. Then it works with `int`, `float`, `Fraction` and even `bool`, but not with complex numbers:

In [ ]:
def safe_sqrt(x):
    if not isinstance(x, numbers.Real):
        raise TypeError(f'a real number is needed, not {type(x).__name__}')
    return math.sqrt(x)


print(safe_sqrt(2), safe_sqrt(Fraction(9, 4)), safe_sqrt(True))
safe_sqrt(1 + 2j)

1.4142135623730951 1.5 1.0


TypeError: a real number is needed, not complex

# 7.  `@dataclass`

### A short introduction to `@dataclass`

Many classes only store data. For such classes, we write the same code again and again: `__init__`, which saves the parameters in attributes, `__repr__` for a readable output, and `__eq__` for comparison. The decorator `@dataclass` from the module [`dataclasses`](https://docs.python.org/3/library/dataclasses.html) creates these methods automatically. It uses the **annotations** of the class attributes: every annotated attribute becomes a **field**.

In [ ]:
from dataclasses import dataclass


@dataclass
class Point:
    x: float
    y: float = 0.0          # a field with a default value


p = Point(1.5)
print(p)                    # __repr__ is created automatically
print(p == Point(1.5, 0))   # __eq__ compares the fields
print(p.x, p.y)

Point(x=1.5, y=0.0)
True
1.5 0.0


Without `@dataclass`, we would write this code by hand:

```python
class Point:
    def __init__(self, x, y=0.0):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'Point(x={self.x!r}, y={self.y!r})'

    def __eq__(self, other):
        if other.__class__ is not self.__class__:
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)
```

Note that annotations are not checked: `Point('a', 'b')` works. Annotations are only hints for the reader and for tools.

The most useful parameters of `@dataclass` and of the function `field`:

| Code | What it does |
|---|---|
| `@dataclass(frozen=True)` | the object cannot be changed after it is created; such objects can be keys of a dictionary |
| `@dataclass(order=True)` | creates `<`, `<=`, `>`, `>=`; the fields are compared in order, like tuples |
| `@dataclass(kw_only=True)` | all the fields become keyword-only parameters of `__init__` (Python 3.10+) |
| `field(default_factory=list)` | every object gets its own new empty list |
| `__post_init__(self)` | a method that `__init__` calls at the end; we use it for checks |

In [ ]:
from dataclasses import field


@dataclass
class Group:
    name: str
    students: list[str] = field(default_factory=list)

    def __post_init__(self):
        if not self.name:
            raise ValueError('The name of a group cannot be empty')


g1 = Group('B-101', ['Anna', 'Ivan'])
g2 = Group('B-102')
g2.students.append('Oleg')
print(g1)
print(g2)
Group('')

Group(name='B-101', students=['Anna', 'Ivan'])
Group(name='B-102', students=['Oleg'])


ValueError: The name of a group cannot be empty

Why do we need `default_factory`? A list as a default value would be **one** list for all objects. `@dataclass` does not allow this mistake:

In [ ]:
try:
    @dataclass
    class BadGroup:
        name: str
        students: list[str] = []
except ValueError as error:
    print('ValueError:', error)

ValueError: mutable default <class 'list'> for field students is not allowed: use default_factory


A frozen dataclass does not allow us to change its fields:

In [ ]:
@dataclass(frozen=True)
class Color:
    r: int
    g: int
    b: int


red = Color(255, 0, 0)
print(red, {red: 'red'})    # a frozen object can be a key of a dictionary
red.g = 100

Color(r=255, g=0, b=0) {Color(r=255, g=0, b=0): 'red'}


FrozenInstanceError: cannot assign to field 'g'

### An abstract dataclass

`@dataclass` and `ABC` work well together. The abstract class describes the **common fields** and the **interface**; the child classes add their own fields and implement the abstract methods. The child classes must also have the decorator `@dataclass`, otherwise their new fields do not appear in `__init__`.

Example: different payment methods. All payments have an amount, and all of them check it in `__post_init__`. The fee depends on the method of payment:

In [ ]:
@dataclass
class Payment(ABC):
    amount: float

    def __post_init__(self):
        if self.amount <= 0:
            raise ValueError('The amount must be positive')

    @abstractmethod
    def fee(self):
        """Return the fee of the payment system."""

    def total(self):
        return self.amount + self.fee()


@dataclass
class CardPayment(Payment):
    card_number: str

    def fee(self):
        return round(self.amount * 0.02, 2)


@dataclass
class CashPayment(Payment):
    def fee(self):
        return 0


payments = [CardPayment(1000, '1234 5678'), CashPayment(300)]
for payment in payments:
    print(payment, '-> total:', payment.total())

CardPayment(amount=1000, card_number='1234 5678') -> total: 1020.0
CashPayment(amount=300) -> total: 300


`__post_init__` is inherited, so the check works for all child classes. And the abstract class still protects itself:

In [ ]:
for create in [lambda: CashPayment(-5), lambda: Payment(100)]:
    try:
        create()
    except (ValueError, TypeError) as error:
        print(f'{type(error).__name__}: {error}')

ValueError: The amount must be positive
TypeError: Can't instantiate abstract class Payment without an implementation for abstract method 'fee'


### The order of fields and `kw_only`

A child dataclass adds its fields **after** the fields of the parent. A field without a default value cannot follow a field with a default value: this is the same rule as for the parameters of a function. Let us add a currency with a default value to the parent class:

In [ ]:
@dataclass
class Payment(ABC):
    amount: float
    currency: str = 'RUB'

    @abstractmethod
    def fee(self):
        """Return the fee of the payment system."""


try:
    @dataclass
    class CardPayment(Payment):
        card_number: str          # no default value, but it follows currency='RUB'

        def fee(self):
            return round(self.amount * 0.02, 2)
except TypeError as error:
    print('TypeError:', error)

TypeError: non-default argument 'card_number' follows default argument 'currency'


The solution is `kw_only=True`. Then all the fields are keyword-only parameters, and their order is not important. The code that creates the objects also becomes easier to read:

In [ ]:
@dataclass(kw_only=True)
class Payment(ABC):
    amount: float
    currency: str = 'RUB'

    @abstractmethod
    def fee(self):
        """Return the fee of the payment system."""


@dataclass(kw_only=True)
class CardPayment(Payment):
    card_number: str

    def fee(self):
        return round(self.amount * 0.02, 2)


card = CardPayment(amount=1000, card_number='1234 5678')
print(card, card.fee())
CardPayment(1000, '1234 5678')    # positional arguments are not allowed now

CardPayment(amount=1000, currency='RUB', card_number='1234 5678') 20.0


TypeError: CardPayment.__init__() takes 1 positional argument but 3 were given

**Summary:**
- `@dataclass` creates `__init__`, `__repr__` and `__eq__` from the annotated fields of a class.
- `field(default_factory=...)` gives a new mutable default value to every object; `__post_init__` is the place for checks.
- `frozen=True` forbids changes; `kw_only=True` makes all the fields keyword-only.
- An abstract dataclass inherits from `ABC` and has the decorator `@dataclass`; the child classes need `@dataclass` too.
- If the parent has fields with default values, use `kw_only=True` (or give default values to the fields of the child).

**Check yourself.** What will the following code print? Think first, then run the cell.

In [ ]:
@dataclass
class Base(ABC):
    x: int

    @abstractmethod
    def double(self):
        """Return 2 * x."""


@dataclass
class Child(Base):
    y: int = 10

    def double(self):
        return 2 * self.x


c = Child(1)
print(c, c.double())
print(c == Child(1, 10), c == Child(1, 11))
print(isinstance(c, Base), Base.__abstractmethods__, Child.__abstractmethods__)

Child(x=1, y=10) 2
True False
True frozenset({'double'}) frozenset()


# Practice

### 0.1. Which objects can we create?

In [ ]:
from abc import ABC, abstractmethod


class Device(ABC):
    @abstractmethod
    def turn_on(self):
        """Turn the device on."""

    @abstractmethod
    def turn_off(self):
        """Turn the device off."""


class Lamp(Device):
    def turn_on(self):
        return 'light'


class SmartLamp(Lamp):
    def turn_off(self):
        return 'dark'


for cls in [Device, Lamp, SmartLamp]:
    try:
        obj = cls()
        print(cls.__name__, 'OK:', obj.turn_on(), obj.turn_off())
    except TypeError:
        print(cls.__name__, 'is abstract:', sorted(cls.__abstractmethods__))

# Your answer:

Device is abstract: ['turn_off', 'turn_on']
Lamp is abstract: ['turn_off']
SmartLamp OK: light dark


### 0.2. A forgotten `ABC`

In [ ]:
from abc import ABC, abstractmethod


class Task:
    @abstractmethod
    def run(self):
        return 'Task.run'


class Report(Task):
    pass


class Job(ABC):
    @abstractmethod
    def run(self):
        return 'Job.run'


class Backup(Job):
    pass


print(Report().run())
print(Backup().run())

# Your answer:

Task.run


TypeError: Can't instantiate abstract class Backup without an implementation for abstract method 'run'

### 0.3. An abstract method with a body

In [ ]:
from abc import ABC, abstractmethod


class Greeting(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def text(self):
        return f'Hello, {self.name}'

    def show(self):
        print(self.text() + '!')


class Polite(Greeting):
    def text(self):
        return super().text() + ', nice to meet you'


class Short(Greeting):
    def text(self):
        return 'Hi'


for greeting in [Polite('Anna'), Short('Ivan')]:
    greeting.show()
print(Greeting.text(Short('Oleg')))

# Your answer:

Hello, Anna, nice to meet you!
Hi!
Hello, Oleg


### 0.4. An abstract dataclass

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass


@dataclass
class Animal(ABC):
    name: str
    legs: int = 4

    @property
    @abstractmethod
    def sound(self):
        """The sound of the animal."""


@dataclass
class Cat(Animal):
    sound = 'meow'


@dataclass
class Bird(Animal):
    legs: int = 2

    @property
    def sound(self):
        return 'tweet'


print(Cat('Murka'))
print(Bird('Kesha'), Bird('Kesha').sound)
print(Cat('Murka') == Cat('Murka', 4), Cat('Murka').sound)
print(Animal('Nobody'))

# Your answer:

Cat(name='Murka', legs=4)
Bird(name='Kesha', legs=2) tweet
True meow


TypeError: Can't instantiate abstract class Animal without an implementation for abstract method 'sound'

### Task 1. Delivery services

1. Create an abstract class `Delivery`:
   - the initializer takes the weight of a parcel in kilograms and saves it in the attribute `weight`;
   - the abstract method `cost()` returns the price of the delivery in rubles;
   - the normal method `info()` returns a string like `'Courier: 3 kg, 450 rub'` (use `type(self).__name__`).
2. Create two concrete classes:
   - `Courier`: the price is `300 + 50 * weight`;
   - `Post`: the price is `200 + 20 * weight`.
3. Check that `Delivery(1)` raises `TypeError`.
4. Create a list of deliveries, print `info()` for each of them and find the cheapest delivery with `min()`.
5. Create the class `Drone(Delivery)` and "forget" to implement `cost()`. What happens when you create an object of this class? Explain the error message.

**Example:**
```python
parcels = [Courier(3), Post(3), Courier(0.5)]
for parcel in parcels:
    print(parcel.info())
# Courier: 3 kg, 450 rub
# Post: 3 kg, 260 rub
# Courier: 0.5 kg, 325.0 rub

cheapest = min(parcels, key=lambda parcel: parcel.cost())
print('The cheapest:', cheapest.info())   # The cheapest: Post: 3 kg, 260 rub

Delivery(1)   # TypeError: Can't instantiate abstract class Delivery without an implementation for abstract method 'cost'
```

In [ ]:
# Your code here

### Task 2. Units of length

1. Create an abstract class `LengthUnit` with two abstract **properties**:
   - `symbol` — the short name of the unit, for example `'km'`;
   - `meters` — the number of meters in one unit, for example `1000` for a kilometer.
2. Add two normal methods to `LengthUnit`:
   - `to_meters(value)` returns `value * self.meters`;
   - `convert(value, other)` converts `value` from this unit into the unit `other` (another object of `LengthUnit`).
3. Create four concrete classes: `Meter` (1 m), `Kilometer` (1000 m), `Mile` (1609.344 m) and `Inch` (0.0254 m). In `Meter` and `Kilometer`, implement the properties with `@property`. In `Mile` and `Inch`, use simple class attributes. Explain why both ways work.
4. Check that `LengthUnit()` raises `TypeError`.

**Example:**
```python
km, mi, inch = Kilometer(), Mile(), Inch()
print(km.to_meters(2.5))                       # 2500.0
print(round(mi.convert(1, km), 3), km.symbol)  # 1.609 km
print(round(inch.convert(12, Meter()), 4))     # 0.3048
print(isinstance(mi, LengthUnit))              # True
```

In [ ]:
# Your code here

### Task 3. Pizza

1. Create an abstract class `Pizza`:
   - the abstract method `toppings()` **has a body**: it returns the base list `['dough', 'tomato sauce', 'cheese']`;
   - the normal method `price()` returns the price: 400 rub for the base and 60 rub for every extra topping (every topping after the first three);
   - the normal method `describe()` returns a string like `'Margherita: dough, tomato sauce, cheese, basil - 460 rub'`.
2. Create the classes `Margherita` (it adds `'basil'`) and `Pepperoni` (it adds `'pepperoni'` and `'chili'`). Their methods `toppings()` must use `super().toppings()` and must not repeat the base list.
3. Answer the questions:
   - Why is `toppings()` abstract if it has a body?
   - Can we create an object of the class `class Plain(Pizza): pass`? How can we create a pizza without extra toppings?

**Example:**
```python
for pizza in [Margherita(), Pepperoni()]:
    print(pizza.describe())
# Margherita: dough, tomato sauce, cheese, basil - 460 rub
# Pepperoni: dough, tomato sauce, cheese, pepperoni, chili - 520 rub
```

In [ ]:
# Your code here

### Task 4. Root finding: a template method

The bisection method and the method of false position (*regula falsi*) find a root of the equation $f(x) = 0$ on an interval $[a, b]$ if $f(a)$ and $f(b)$ have different signs. The two algorithms are the same. They differ only in the way they choose the next point $c$ inside $[a, b]$:
- the bisection method: $c = \dfrac{a + b}{2}$;
- the method of false position: $c = b - f(b)\,\dfrac{b - a}{f(b) - f(a)}$ (the point where the chord through $(a, f(a))$ and $(b, f(b))$ crosses the axis $x$).

1. Create an abstract class `RootFinder`:
   - the initializer takes `tol=1e-8` and `max_iter=100` and saves them;
   - the abstract method `next_point(f, a, b)` returns the next point $c$;
   - the template method `solve(f, a, b)`:
     1. if $f(a) \cdot f(b) > 0$, it raises `ValueError('f(a) and f(b) must have different signs')`;
     2. it repeats at most `max_iter` times: `c = self.next_point(f, a, b)`; if $|f(c)| <$ `tol`, the loop stops; otherwise it keeps the half of the interval where $f$ changes its sign (if $f(a) \cdot f(c) < 0$, then `b = c`, else `a = c`);
     3. it returns a tuple `(c, number of iterations)`.
2. Create the concrete classes `Bisection` and `FalsePosition`. Each of them implements only `next_point`.
3. Compare the two methods for $f(x) = x^2 - 2$ on $[0, 2]$ and for $f(x) = \cos x$ on $[0, 3]$. Which method needs fewer iterations?

**Example:**
```python
import math

for finder in [Bisection(), FalsePosition()]:
    root, iterations = finder.solve(lambda x: x ** 2 - 2, 0, 2)
    print(f'{type(finder).__name__:>13}: x = {root:.8f}, iterations = {iterations}')
#     Bisection: x = 1.41421356, iterations = 28
# FalsePosition: x = 1.41421356, iterations = 12

Bisection().solve(math.cos, 0, 1)   # ValueError: f(a) and f(b) must have different signs
```

In [ ]:
# Your code here

### Task 5. Records in a text file

1. Create an abstract class `Record` with:
   - the abstract **class method** `from_line(cls, line)` — it creates an object from one line of text;
   - the abstract method `to_line()` — the reverse operation: it returns the object as one line;
   - the normal class method `from_text(cls, text)` — it splits the text into lines, skips the empty lines and the lines that start with `#`, and returns a list of objects (use `cls.from_line`).
2. Create two **dataclasses** that inherit from `Record`:
   - `Student` with the fields `name: str`, `group: str`, `grade: int`; the format of a line is `'Anna;B-101;5'`. The method `__post_init__` checks that the grade is from 1 to 5; otherwise, it raises `ValueError`;
   - `Book` with the fields `title: str`, `year: int`; the format of a line is `'War and Peace (1869)'`.

**Hint:** `'War and Peace (1869)'.rsplit(' (', 1)` returns `['War and Peace', '1869)']`.

**Example:**
```python
text = """
# name;group;grade
Anna;B-101;5

Ivan;B-102;4
"""
students = Student.from_text(text)
print(students)
# [Student(name='Anna', group='B-101', grade=5), Student(name='Ivan', group='B-102', grade=4)]
print(students[0].to_line())                    # Anna;B-101;5
book = Book.from_line('War and Peace (1869)')
print(book, book.to_line())                     # Book(title='War and Peace', year=1869) War and Peace (1869)
Student.from_line('Oleg;B-101;7')               # ValueError: The grade must be from 1 to 5
```

In [ ]:
# Your code here

### Task 6. Train tickets

1. Create an abstract dataclass `Ticket` with the fields `passenger: str`, `base_price: float` and `currency: str = 'RUB'`:
   - `__post_init__` raises `ValueError` if `base_price` is not positive;
   - the abstract method `price()` returns the final price;
   - the normal method `info()` returns a string like `'Anna: 3000 RUB'`.
2. Create three child dataclasses:
   - `EconomyTicket`: the price is `base_price`;
   - `StudentTicket` with the field `student_id: str`: the price is 50% of `base_price`;
   - `BusinessTicket` with the field `lounge: bool = True`: the price is `2 * base_price`, plus 1500 if `lounge` is `True`.
3. First, write `StudentTicket` without `kw_only`. Which error appears and why? Then fix it with `kw_only=True`.
4. A sold ticket must not change. Make all the tickets `frozen=True` and check that `ticket.base_price = 1` raises an error. (Note: a frozen dataclass cannot inherit from a non-frozen one, and the other way round.)

**Example:**
```python
tickets = [
    EconomyTicket(passenger='Anna', base_price=3000),
    StudentTicket(passenger='Ivan', base_price=3000, student_id='S-17'),
    BusinessTicket(passenger='Oleg', base_price=3000, lounge=False),
]
for ticket in tickets:
    print(ticket.info())
# Anna: 3000 RUB
# Ivan: 1500.0 RUB
# Oleg: 6000 RUB

print(tickets[1])
# StudentTicket(passenger='Ivan', base_price=3000, currency='RUB', student_id='S-17')
tickets[0].base_price = 1                       # FrozenInstanceError
Ticket(passenger='Nobody', base_price=100)      # TypeError
EconomyTicket(passenger='Nobody', base_price=0) # ValueError
```

In [ ]:
# Your code here

### Task 7. An arithmetic progression

1. Create the class `ArithmeticProgression(start, step, n)` as a child of `collections.abc.Sequence`. It describes the first `n` terms of the progression `start, start + step, start + 2*step, ...`. The class must **not** store the terms in a list.
2. Implement only the methods `__len__` and `__getitem__`. `__getitem__` supports negative indices and raises `IndexError` for an index outside the sequence.
3. Check that `in`, the loop `for`, `reversed()`, `index()` and `count()` work, although you have not written them.
4. What happens if you delete the method `__len__`? Check it.
5. ⭐ Add the support of slices: `ap[1:4]` returns a new `ArithmeticProgression`. **Hint:** for a slice `s`, the call `range(len(self))[s]` gives a `range` with the right `start`, `step` and length.

**Example:**
```python
ap = ArithmeticProgression(start=1, step=3, n=5)   # 1, 4, 7, 10, 13
print(len(ap), ap[0], ap[-1])     # 5 1 13
print(list(ap))                   # [1, 4, 7, 10, 13]
print(7 in ap, 8 in ap)           # True False
print(list(reversed(ap)))         # [13, 10, 7, 4, 1]
print(ap.index(10), ap.count(4))  # 3 1
ap[5]                             # IndexError

# ⭐ slices
print(list(ap[1:4]))              # [4, 7, 10]
print(list(ap[::-2]))             # [13, 7, 1]
```

In [ ]:
# Your code here

### Task 8. Any numbers in any containers

Write a function `flatten_numbers(data)`. It returns a flat list of all the numbers in `data`:
- if `data` is a real number (`numbers.Real`: `int`, `float`, `Fraction`, ...), the function returns `[data]`;
- if `data` is a number, but not a real number (for example, `complex` or `Decimal`), it raises `TypeError` with the message `'<type> is not a real number'`;
- if `data` is iterable (`collections.abc.Iterable`), but not a string, the function processes all its items recursively (lists, tuples, sets, `range`, generators, the class `Squares` from the seminar...);
- in all other cases (including strings), it raises `TypeError` with the message `'unsupported type <type>'`.

Use `isinstance` **only with abstract classes** from `numbers` and `collections.abc` (and with `str`), not with `list`, `tuple`, `int` or `float`. Then use the function to write `mean(data)`, which returns the arithmetic mean of all the numbers.

**Example:**
```python
from fractions import Fraction
from decimal import Decimal

print(flatten_numbers([1, 2.5, (Fraction(1, 2), [3])]))   # [1, 2.5, Fraction(1, 2), 3]
print(flatten_numbers(range(3)))                          # [0, 1, 2]
print(flatten_numbers(Squares(4)))                        # [0, 1, 4, 9]
print(flatten_numbers({'a': 1}.values()))                 # [1]
print(mean([1, [2, 3], (x for x in [4, 5])]))             # 3.0
flatten_numbers([1, 2j])                 # TypeError: complex is not a real number
flatten_numbers([Decimal('1.5')])        # TypeError: Decimal is not a real number
flatten_numbers(['abc'])                 # TypeError: unsupported type str
```

In [ ]:
# Your code here

### Task 9 ⭐. Objects that can be reset

1. Create an abstract class `Resettable` with the abstract method `reset()`.
2. Add the class method `__subclasshook__`: every class with a **callable** attribute `reset` is a subclass of `Resettable`. Do not forget the check `cls is Resettable`.
3. Create three classes that do **not** inherit from `Resettable`:
   - `Counter` with the attribute `value` and the methods `increase()` and `reset()`;
   - `Timer` with the attribute `seconds` and the method `reset()`;
   - `Note` with the attribute `text` and **without** `reset()`.
4. Write the function `reset_all(objects)`. It calls `reset()` for every object that is `Resettable`, skips the other objects, and returns the number of objects that were reset.
5. Check: `issubclass(list, Resettable)` returns `False`. What does `issubclass` return for a class that has the class attribute `reset = 0`?

**Example:**
```python
counter = Counter()
counter.increase()
counter.increase()
timer = Timer(seconds=42)
note = Note('buy milk')

print(isinstance(counter, Resettable), isinstance(note, Resettable))   # True False
print(reset_all([counter, timer, note]))                               # 2
print(counter.value, timer.seconds, note.text)                         # 0 0 buy milk
```

In [ ]:
# Your code here